In [ ]:
from scipy.stats import norm
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# Reproducible samples
rng = np.random.default_rng(0)

n = 1000
# First distribution: 2D Gaussian, mean (-10, 0), variance 1
mean0 = np.array([-10.0, 0.0])
std0 = 1.0  # variance 1 -> std 1
samples0 = rng.normal(loc=mean0, scale=std0, size=(n, 2))

# Second distribution: 2D Gaussian, mean (10, 0), variance 1
mean1 = np.array([10.0, 0.0])
std1 = 1.0  # variance 1 -> std 1
rand_samples1 = True
if rand_samples1:
    samples1 = rng.normal(loc=mean1, scale=std1, size=(n, 2))
else:
    samples1 = samples0.copy()
    # samples1[:, 0] *= -1
    samples1[:, 0] += 20

# For each pair (samples1[i], samples2[i]), sample 1000 evenly spaced points
# on the line segment between them using t in [0, 1].
# Result: 1,000,000 points total.
time_step = 1000
t = np.linspace(0.0, 1.0, time_step, endpoint=True).reshape(1, time_step, 1)
line_points = (1.0 - t) * samples1[:, None, :] + t * samples0[:, None, :]
line_points = line_points.reshape(-1, 2)

time_step = 100
sig_min = 1 / time_step / 10
trajectory = np.repeat(samples0[:, :, None], time_step + 1, axis=2)

#xmin, xmax = 6, 14
# ymin, ymax = -4, 4
#
# nx, ny = 100, 100  # number of points along x and y
#
# xs = np.linspace(xmin, xmax, nx)
# ys = np.linspace(ymin, ymax, ny)
#
# X, Y = np.meshgrid(xs, ys)          # shapes (ny, nx)
# points = np.column_stack([X.ravel(), Y.ravel()])  # shape (nx*ny, 2)
points = samples0.copy()
points[:, 0] += 20
p_x1 = norm.pdf(points, loc=mean1, scale=std1).prod(axis=1)

i = 0
for t in tqdm(np.linspace(0.0, 1.0, time_step, endpoint=False)):
    # mean and std of pt(x|x1)
    mu_p_t_x_x1 = t * points + (1 - t) * mean0
    sig_p_t_x_x1 = 1 - (1 - sig_min) * t

    # probability of pt(x|x1)
    p_t_x_x1 = norm.pdf(trajectory[:, :, i][:, None, :], loc=mu_p_t_x_x1[None, :, :], scale=sig_p_t_x_x1).prod(axis=-1)
    # ut(x|x1) = (x1 - x) / (1 - t)
    cond_velocity = (points[None, :, :] - trajectory[:, :, i][:, None, :]) / (1 - (1 - sig_min) * t)
    # ut(x) = \int ut(x|x1) * pt(x|x1) * p(x1) / pt(x) dx1
    weights = p_t_x_x1 * p_x1
    velocity = (cond_velocity * p_t_x_x1[:, :, None]).sum(1) / p_t_x_x1.sum(1)[:, None]
    # velocity = (cond_velocity * weights[:, :, None]).sum(1) / weights.sum(1)[:, None]

    trajectory[:, :, i + 1] = trajectory[:, :, i] + velocity / time_step
    i += 1
fig, axes = plt.subplots(2, 1, figsize=(10, 16))

axes[0].scatter(samples0[:, 0], samples0[:, 1], s=12, alpha=1, edgecolors='none')
axes[0].scatter(samples1[:, 0], samples1[:, 1], s=12, alpha=1, edgecolors='none')
axes[0].scatter(line_points[:, 0], line_points[:, 1], s=3, alpha=0.1, edgecolors='none')
axes[0].grid(True, linestyle='--', linewidth=0.2, alpha=0.2)
axes[0].set_title('Conditional trajectories')

# left: trajectories using 'trajectory'
axes[1].scatter(samples0[:, 0], samples0[:, 1], s=12, alpha=1, edgecolors='none', label='x0')
axes[1].scatter(samples0[:, 0] + 20, samples0[:, 1], s=12, alpha=1, edgecolors='none', label='x1')
axes[1].grid(True, linestyle='--', linewidth=.5, alpha=.4)
# plot trajectories for all points
for i in tqdm(range(n)):
    # left panel: trajectory
    traj = trajectory[i]           # shape (2, time_step+1)
    x = traj[0]
    y = traj[1]
    axes[1].plot(x, y, linewidth=.5, alpha=.5)
axes[1].set_title('Marginal trajectories')

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(trajectory[:, :, -1][:, 0], trajectory[:, :, -1][:, 1], s=12, alpha=1, edgecolors='none')
ax.scatter(samples0[:, 0] + 20, samples0[:, 1], s=12, alpha=1, edgecolors='none')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ---------------------------------------------------
# parameters
# ---------------------------------------------------

np.random.seed(0)

mu = 10.0
m0 = np.array([-mu, 0.0])
m1 = np.array([mu, 0.0])

N_TRAJ = 1000
T = np.linspace(0,1,300)

# ---------------------------------------------------
# marginal vector field
# ---------------------------------------------------

def u_t(t, x):

    x = np.asarray(x)

    mt = (1-t)*m0 + t*m1
    sigma2 = (1-t)**2 + t**2

    term1 = (m1 - m0)
    term2 = ((2*t - 1) / sigma2) * (x - mt)

    return term1 + term2


# ---------------------------------------------------
# integrate ODE trajectory
# ---------------------------------------------------

def integrate(x0):

    sol = solve_ivp(
        lambda t,x: u_t(t,x),
        (0,1),
        x0,
        t_eval=T,
        rtol=1e-6,
        atol=1e-6
    )

    return sol.y


# ---------------------------------------------------
# sample starting points (used everywhere)
# ---------------------------------------------------

x0_samples = np.random.randn(N_TRAJ,2) + m0

trajectories = []

for x0 in x0_samples:
    traj = integrate(x0)
    trajectories.append(traj)

trajectories = np.array(trajectories)

x1_samples = trajectories[:, :, -1]

# ---------------------------------------------------
# plotting
# ---------------------------------------------------

plt.figure(figsize=(10,8))

# trajectories
for traj in trajectories:
    plt.plot(traj[0], traj[1], alpha=0.8)

# use same data points
plt.scatter(x0_samples[:,0], x0_samples[:,1], s=30, label="p0 samples")
plt.scatter(x1_samples[:,0], x1_samples[:,1], s=30, label="p1 samples")

plt.title("Marginal paths $\\phi_t(x_0)$")
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# mixture means
mu0 = np.array([[-10, -10],
                [-10,  10]])

mu1 = np.array([[10, -10],
                [10,  10]])

# interpolation mean
def m_kj(k, j, t):
    return (1 - t) * mu0[k] + t * mu1[j]

def sigma2(t):
    return t**2 + (1 - t)**2


# gaussian density
def gaussian(x, mean, var):
    diff = x - mean
    return np.exp(-0.5 * np.sum(diff**2, axis=-1) / var) / (2*np.pi*var)


# mixture density
def pt_density(grid_points, t):

    s = sigma2(t)
    total = np.zeros(grid_points.shape[:2])

    for k in range(2):
        for j in range(2):
            m = m_kj(k, j, t)
            total += 0.25 * gaussian(grid_points, m, s)

    return total


# responsibility weights
def weights(x, t):

    s = sigma2(t)
    logvals = np.zeros((2,2))

    for k in range(2):
        for j in range(2):

            m = m_kj(k, j, t)
            d = x - m
            logvals[k,j] = -0.5*np.dot(d,d)/s

    maxlog = np.max(logvals)
    vals = np.exp(logvals - maxlog)
    vals /= vals.sum()

    return vals


# marginal velocity
def velocity(t, x):

    w = weights(x, t)
    v = np.zeros(2)

    for k in range(2):
        for j in range(2):

            m = m_kj(k, j, t)

            v += w[k,j] * (
                mu1[j] - mu0[k]
                + (2*t - 1)/(t**2 + (1-t)**2) * (x - m)
            )

    return v


# sample starting particles
np.random.seed(0)
N = 300

x0 = np.vstack([
    mu0[0] + np.random.randn(N//2,2),
    mu0[1] + np.random.randn(N//2,2)
])


# integrate trajectories
t_eval = np.linspace(0,1,120)
paths = []

for p in x0:

    sol = solve_ivp(
        lambda t,y: velocity(t,y),
        [0,1],
        p,
        t_eval=t_eval
    )

    paths.append(sol.y.T)

paths = np.array(paths)


# grid for density
grid = np.linspace(-15,15,120)
X, Y = np.meshgrid(grid, grid)
grid_points = np.stack([X,Y], axis=-1)


# plotting
fig, ax = plt.subplots(figsize=(7,7))

def update(frame):

    ax.clear()

    t = t_eval[frame]

    Z = pt_density(grid_points, t)

    ax.contourf(X, Y, Z, levels=25, cmap="magma", alpha=0.7)

    for p in paths:
        ax.plot(p[:frame,0], p[:frame,1], color="white", alpha=0.15)

    ax.scatter(paths[:,frame,0],
               paths[:,frame,1],
               s=6,
               color="cyan")

    ax.scatter(mu0[:,0], mu0[:,1],
               c="blue",
               s=160,
               label="source")

    ax.scatter(mu1[:,0], mu1[:,1],
               c="red",
               s=160,
               label="target")

    ax.set_xlim(-15,15)
    ax.set_ylim(-15,15)
    ax.set_aspect("equal")

    ax.set_title(f"Flow Matching Transport   t={t:.2f}")
    ax.legend()

anim = FuncAnimation(fig, update, frames=len(t_eval), interval=60)

HTML(anim.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 50  # MB
plt.style.use("dark_background")

# --------------------------------------------------
# Gaussian mixture means
# --------------------------------------------------

mu0 = np.array([[-10,-10],
                [-10,10]])

mu1 = np.array([[10,-10],
                [10,10]])

# --------------------------------------------------
# helpers
# --------------------------------------------------

def sigma2(t):
    return t**2 + (1-t)**2

def m_kj(k,j,t):
    return (1-t)*mu0[k] + t*mu1[j]

# --------------------------------------------------
# density
# --------------------------------------------------

def gaussian(x, mean, var):

    diff = x - mean
    return np.exp(-0.5*np.sum(diff**2,axis=-1)/var)/(2*np.pi*var)


def pt_density(grid,t):

    s = sigma2(t)
    z = np.zeros(grid.shape[:2])

    for k in range(2):
        for j in range(2):
            m = m_kj(k,j,t)
            z += 0.25 * gaussian(grid,m,s)

    return z


# --------------------------------------------------
# responsibility weights
# --------------------------------------------------

def weights(x,t):

    s = sigma2(t)

    logv = np.zeros((2,2))

    for k in range(2):
        for j in range(2):

            m = m_kj(k,j,t)
            d = x-m
            logv[k,j] = -0.5*np.dot(d,d)/s

    logv -= np.max(logv)

    v = np.exp(logv)
    v /= v.sum()

    return v


# --------------------------------------------------
# velocity field
# --------------------------------------------------

def velocity(t,x):

    w = weights(x,t)

    v = np.zeros(2)

    for k in range(2):
        for j in range(2):

            m = m_kj(k,j,t)

            v += w[k,j]*(
                mu1[j]-mu0[k]
                + (2*t-1)/(t**2+(1-t)**2)*(x-m)
            )

    return v


# --------------------------------------------------
# sample particles
# --------------------------------------------------

np.random.seed(1)

N = 200

x0 = np.vstack([
    mu0[0] + np.random.randn(N//2,2),
    mu0[1] + np.random.randn(N//2,2)
])

# --------------------------------------------------
# integrate trajectories
# --------------------------------------------------

t_eval = np.linspace(0,1,100)

paths = []

for p in x0:

    sol = solve_ivp(
        lambda t,y: velocity(t,y),
        [0,1],
        p,
        t_eval=t_eval
    )

    paths.append(sol.y.T)

paths = np.array(paths)


# --------------------------------------------------
# grid for density + velocity
# --------------------------------------------------

grid = np.linspace(-15,15,80)

X,Y = np.meshgrid(grid,grid)

grid_pts = np.stack([X,Y],axis=-1)


# --------------------------------------------------
# animation
# --------------------------------------------------

fig,ax = plt.subplots(figsize=(8,8))

def update(frame):

    ax.clear()

    t = t_eval[frame]

    # density
    Z = pt_density(grid_pts,t)

    ax.contourf(
        X,Y,Z,
        levels=40,
        cmap="inferno",
        alpha=0.85
    )

    # streamlines
    U = np.zeros_like(X)
    V = np.zeros_like(Y)

    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            v = velocity(t,np.array([X[i,j],Y[i,j]]))
            U[i,j] = v[0]
            V[i,j] = v[1]

    ax.streamplot(
        X,Y,U,V,
        color="white",
        density=1.1,
        linewidth=0.7,
        arrowsize=0.7
    )

    # trajectories
    for p in paths:
        ax.plot(p[:frame,0],p[:frame,1],
                color="cyan",
                alpha=0.25,
                linewidth=1)

    # particles
    ax.scatter(paths[:,frame,0],
               paths[:,frame,1],
               s=8,
               color="cyan")

    # means
    ax.scatter(mu0[:,0],mu0[:,1],
               c="dodgerblue",
               s=220,
               label="source")

    ax.scatter(mu1[:,0],mu1[:,1],
               c="red",
               s=220,
               label="target")

    ax.set_xlim(-15,15)
    ax.set_ylim(-15,15)
    ax.set_aspect("equal")

    ax.set_title(f"Flow Matching Transport   t={t:.2f}",fontsize=16)

    ax.legend(loc="upper left")

anim = FuncAnimation(fig,update,frames=len(t_eval),interval=70)

HTML(anim.to_jshtml())